# Phase 1 Check

In [ ]:
import sys
import os

# Set working directory to project root (/home/jovyan/work)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
os.chdir(project_root)
if project_root not in sys.path:
    sys.path.append(project_root)

# Imports
from src.data.ingestion import MarketDataIngestion
from src.data.spark_pipeline import get_spark_session, SparkTechnicalIndicators
from src.data.databricks_client import DatabricksClient

print("Imports successful!")

Imports successful!


In [5]:
# 1. Fetch OHLCV
fetcher = MarketDataIngestion()
pandas_df = fetcher.fetch_daily_ohlcv("AAPL")

# 2. Spark Transformations
spark = get_spark_session()
spark_df = spark.createDataFrame(pandas_df)

indicator_calc = SparkTechnicalIndicators(spark)
processed_df = indicator_calc.compute_indicators(spark_df)

# 3. Store Data
db_client = DatabricksClient()
db_client.write_dataset(processed_df, table_name="aapl_indicators")

print("Phase 1 Execution Complete!")

[2026-09-22 16:14:48] [INFO] [stonks_maker]: Fetching OHLCV for AAPL via yfinance...
[2026-09-22 16:14:48] [INFO] [stonks_maker]: Starting Spark indicator transformations...
[2026-09-22 16:14:49] [INFO] [stonks_maker]: Completed Spark indicator calculations.
[2026-09-22 16:14:49] [INFO] [stonks_maker]: Databricks credentials not configured. Saving locally to Parquet.


PermissionError: [Errno 13] Permission denied: 'data'

In [ ]:
# Verify saved Parquet/Delta dataset
output_df = db_client.read_dataset(spark, table_name="aapl_indicators")
output_df.show(5)

[2026-09-22 15:54:23] [INFO] [stonks_maker]: Reading dataset locally from data/processed/aapl_indicators
+----------+------+------------------+------------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+------------------+
|      date|ticker|              open|              high|               low|             close|   volume|            sma_20|            sma_50|   bollinger_upper|   bollinger_lower|            rsi_14|
+----------+------+------------------+------------------+------------------+------------------+---------+------------------+------------------+------------------+------------------+------------------+
|2025-09-22|  AAPL|247.38616206443075|255.69547916115815| 247.2068166293636|255.13751220703125|105517400|255.13751220703125|255.13751220703125|              NULL|              NULL|               0.0|
|2025-09-23|  AAPL| 254.9382669065294| 256.3928850249964|252.64672876089196|253.49359130859

# Phase 2 Check